# 10i NSD-synthetic with DINOv2 features

**Kaggle:** GPU P100, Internet on. **Inputs:** the outputs of 10a and 10b.

Development analysis, not a prespecified test. 10h with 10b's DINOv2 features instead of AlexNet. Does a modern self-supervised model carry borrowing and the
map to unusual images? The DINOv2 kernel PCA is refitted on the GPU and mapped onto 10b's components (the run stops
unless R^2 >= 0.99).

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '10i-nsdsyndino', 'gpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"K": 200, "ks_primary": [250, 500], "ks_descriptive": [100, 1000], "draws": [13, 14, 15], "folds": 5, "n_galleries": 4, "gallery_size": 200, "gallery_seed": 346, "delta_r_share": 0.05, "delta_top1": 2.0, "n_claims": 3, "alpha": 0.05, "wall_hours": 11.0, "roi_classes": ["prf-visualrois", "floc-faces", "floc-places", "floc-bodies", "floc-words"], "k": 500, "half_seed": 386, "n_images": 284, "n_shared": 220, "n_color": 64, "n_trials": 744, "beta_version": "betas_fithrf_GLMdenoise_RR", "image_size": 256, "jpeg_quality": 95, "fit_images": 16540, "n_components": 1000, "batch": 64, "kpca_chunk": 4096, "min_align_r2": 0.99, "ridge_seed": 13, "feature": "dinov2", "dinov2_layers": [4, 8, 12], "dinov2_pool": 7, "dinov2_batch": 64}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import sys
import numpy as np
import pandas as pd
import torch
if not torch.cuda.is_available():
    raise RuntimeError("10i needs the GPU (DINOv2 features)")
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

if not isinstance(FIXED["K"], int):
    raise RuntimeError("K is not fixed: fill it in from 09q before the confirmation runs")
IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
shared = np.load(conv / "shared_train_nsd_ids.npy")
cols, common = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
ids, Y = {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)
noise = np.concatenate([np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]) for s in SUBJECTS])
DELTA_R = FIXED["delta_r_share"] * float(np.median(np.sqrt(np.clip(noise, 0, None) / 100)))
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]]
test_ids = np.sort(shared)
rows_of = {s: {int(i): j for j, i in enumerate(ids[s])} for s in SUBJECTS}
test_rows = {s: np.array([rows_of[s][int(i)] for i in test_ids]) for s in SUBJECTS}
pool_rows = {s: np.setdiff1d(np.arange(len(ids[s])), test_rows[s]) for s in SUBJECTS}
GALS = nc.galleries(len(test_ids), FIXED["n_galleries"], FIXED["gallery_size"], FIXED["gallery_seed"])
rois = {}
for cls in FIXED["roi_classes"]:
    for s in SUBJECTS:
        lab = np.concatenate([np.load(next((conv / f"{s}_roi").rglob(f"{h}.{cls}_challenge_space.npy")))[cols[h][s]] for h in ("lh", "rh")])
        rois[(s, cls)] = lab
print(f"{V} common vertices; delta_r {DELTA_R:.4f}; {len(test_ids)} test images; pools", {s: len(pool_rows[s]) for s in SUBJECTS},
      "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
import io
import subprocess
import urllib.request
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel", "h5py"], check=True)
import h5py
import nibabel as nib
import scipy.io as sio
from PIL import Image
from cross_subject_encoding.features import streamed
from cross_subject_encoding.models import paper_linearizing as pl
TMP = Path("/kaggle/tmp/nsdsyn")
TMP.mkdir(parents=True, exist_ok=True)

def fetch(url, dst, tries=5):
    for i in range(tries):
        try:
            urllib.request.urlretrieve(url, dst)
            return dst
        except Exception as e:
            if i == tries - 1:
                raise
            print("  retry", url.split("/")[-1], type(e).__name__)
            time.sleep(15 * (i + 1))

order = sio.loadmat(fetch(f"{C.NSD_AWS}/nsddata/experiments/nsdsynthetic/nsdsynthetic_expdesign.mat", TMP / "d.mat"))["masterordering"].ravel().astype(int) - 1
if len(order) != FIXED["n_trials"] or order.min() != 0 or order.max() != FIXED["n_images"] - 1:
    raise ValueError("unexpected NSD-synthetic design")
reps = np.bincount(order, minlength=FIXED["n_images"])
print("trials", len(order), "| repetitions per image", dict(zip(*map(lambda a: a.tolist(), np.unique(reps, return_counts=True)))))

def brick(path, n):
    with h5py.File(path, "r") as f:
        keys = [k for k in f.keys() if isinstance(f[k], h5py.Dataset)]
        a = np.asarray(f[keys[0]])
    # to (images, height, width, 3) whatever the stored axis order
    ax_n, ax_c = a.shape.index(n), a.shape.index(3)
    rest = [i for i in range(a.ndim) if i not in (ax_n, ax_c)]
    a = a.transpose([ax_n] + rest + [ax_c])
    if a.shape[2] < a.shape[1]:  # width must be the longer side (1360 x 714 screen)
        a = a.transpose(0, 2, 1, 3)
    return a

syn_dir = C.PROJECT_ROOT / "nsdsyn_images"
syn_dir.mkdir(exist_ok=True)
def save_crops(a, names):
    h, w = a.shape[1:3]
    x0 = (w - h) // 2
    for img, name in zip(a, names):
        Image.fromarray(img[:, x0:x0 + h]).resize((FIXED["image_size"],) * 2, Image.BICUBIC).save(syn_dir / name, quality=FIXED["jpeg_quality"])
shared_names = [f"shared_{i:03d}.jpg" for i in range(FIXED["n_shared"])]
a = brick(fetch(f"{C.NSD_AWS}/nsddata_stimuli/stimuli/nsdsynthetic/nsdsynthetic_stimuli.hdf5", TMP / "s.hdf5"), FIXED["n_shared"])
print("shared stimuli", a.shape, a.dtype)
save_crops(a, shared_names)
del a
(TMP / "s.hdf5").unlink()
SYN_NAMES, SYN_Y = {}, {}
for s in SUBJECTS:
    a = brick(fetch(f"{C.NSD_AWS}/nsddata_stimuli/stimuli/nsdsynthetic/nsdsynthetic_colorstimuli_{s}.hdf5", TMP / "c.hdf5"), FIXED["n_color"])
    color_names = [f"{s}_color_{j:02d}.jpg" for j in range(FIXED["n_color"])]
    save_crops(a, color_names)
    del a
    SYN_NAMES[s] = shared_names + color_names  # image index i (0-based) -> file; 220.. are this person's colour images
    parts = []
    for h in ("lh", "rh"):
        f = fetch(f"{C.NSD_AWS}/nsddata_betas/ppdata/{s}/fsaverage/nsdsyntheticbetas_{FIXED['beta_version'].split('betas_', 1)[1]}/{h}.betas_nsdsynthetic.mgh", TMP / f"{h}.mgh")
        b = np.asarray(nib.load(str(f)).dataobj, dtype=np.float32).reshape(-1, FIXED["n_trials"])
        parts.append(b[common[h]].T)  # the common challenge vertices, in 10c's order (fsaverage indices)
        f.unlink()
    SYN_Y[s] = np.concatenate(parts, 1)
    if SYN_Y[s].shape != (FIXED["n_trials"], V) or not np.isfinite(SYN_Y[s]).all():
        raise ValueError(f"{s}: synthetic betas {SYN_Y[s].shape}")
    print(s, "synthetic betas", SYN_Y[s].shape, "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

# features: 10b's fit images plus every synthetic image, kernel PCA refitted, then mapped onto 10b's components
f10b = np.load(next(IN.rglob("10b_dinov2_kpca1000.npz")))
images_dir = next(p for p in IN.rglob("images") if p.is_dir() and any(p.glob("nsd-*.jpg")))
all_paths = sorted(images_dir.glob("nsd-*.jpg"))
if not np.array_equal(np.array([int(p.stem.split("-")[1]) for p in all_paths]), f10b["nsd_ids"]):
    raise ValueError("image list differs from 10b's")
fit_paths = [str(all_paths[i]) for i in f10b["fit_idx"]]
syn_files = sorted({n for s in SUBJECTS for n in SYN_NAMES[s]})
paths = fit_paths + [str(syn_dir / n) for n in syn_files]
n_fit = len(fit_paths)
from cross_subject_encoding.features import modern as mf
z_new, info = streamed.streamed_kpca(lambda rows: mf.extract([paths[r] for r in rows], "dinov2", "cuda", None, FIXED["dinov2_batch"],
                                                             tuple(FIXED["dinov2_layers"]), FIXED["dinov2_pool"]),
                                     len(paths), np.arange(n_fit), "cuda", FIXED["n_components"], FIXED["kpca_chunk"])
z_old = f10b["z"][f10b["fit_idx"]].astype(np.float64)
W = np.linalg.lstsq(z_new[:n_fit].astype(np.float64), z_old, rcond=None)[0]
align_r2 = 1 - ((z_old - z_new[:n_fit] @ W) ** 2).sum(0) / ((z_old - z_old.mean(0)) ** 2).sum(0)
print("alignment R^2 per component: min", round(float(align_r2.min()), 5), "median", round(float(np.median(align_r2)), 5), "|",
      f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
if align_r2.min() < FIXED["min_align_r2"]:
    raise ValueError("the refitted kernel PCA does not reproduce 10b's components")
z_syn_all = z_new[n_fit:].astype(np.float64) @ W
row_of = {n: j for j, n in enumerate(syn_files)}
Z_SYN = {s: z_syn_all[[row_of[n] for n in SYN_NAMES[s]]] for s in SUBJECTS}
np.savez(C.PROJECT_ROOT / "10i_nsdsyn_dinov2_kpca1000.npz", names=np.array(syn_files), z=z_syn_all, align_r2=align_r2)

In [ ]:
def ols_coef(x, y):
    # the paper's OLS (intercept, normal equations), per output, in float64
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
BETA = {}
for name in ("dinov2",):
    for s in SUBJECTS:
        BETA[(name, s)] = ols_coef(feats(name, ids[s][pool_rows[s]]), Y[s][pool_rows[s]])
    print(name, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")

def ridge_coef(x, y):
    m = RidgePerOutput(lm.ALPHAS, FIXED["folds"], FIXED["ridge_seed"]).fit(x, np.asarray(y, np.float32))
    print("   alpha median", float(np.median(m.best_alpha_)), "at grid edge", round(m.alpha_at_edge_, 3), flush=True)
    return np.vstack([(m.y_mean_ - m.x_mean_ @ m.coef_)[None], m.coef_]).astype(np.float32)
BETA_R = {}
for s in SUBJECTS:
    BETA_R[("dinov2", s)] = ridge_coef(feats("dinov2", ids[s][pool_rows[s]]), Y[s][pool_rows[s]])
    print(s, "ridge encoder |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
def r_vert(p, m):
    p, m = p - p.mean(0), m - m.mean(0)
    return (p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)

def by_image(y):
    # session z-score per vertex, then each image's mean, and the ceiling of those means (unequal repetitions)
    y = (y - y.mean(0)) / (y.std(0) + 1e-12)
    mean = np.stack([y[order == i].mean(0) for i in range(FIXED["n_images"])])
    ss = sum(((y[order == i] - mean[i]) ** 2).sum(0) for i in range(FIXED["n_images"]))
    noise = ss / (len(order) - FIXED["n_images"])
    snr2 = np.clip(1 - noise, 0, None) / np.maximum(noise, 1e-12)
    return mean, snr2

def ceiling(snr2, imgs):
    return np.sqrt(snr2 / (snr2 + np.mean(1 / reps[imgs])))

from scipy import stats
# how far each image lies from the natural images in feature space: the distance to its nearest natural pool image
# (person's own pool), divided by the median nearest-neighbour distance among the natural test images
def nn_dist(q, ref, chunk=512):
    ref2 = (ref ** 2).sum(1)
    return np.concatenate([np.sqrt(np.clip((q[i:i + chunk] ** 2).sum(1)[:, None] - 2 * q[i:i + chunk] @ ref.T + ref2[None], 0, None)).min(1)
                           for i in range(0, len(q), chunk)])
FEAT_DIST, FEAT_DIST_NAT, NAT_NC, feat_rows = {}, {}, {}, []
for s in SUBJECTS:
    ref = feats("dinov2", ids[s][pool_rows[s]]).astype(np.float64)
    d_nat = nn_dist(feats("dinov2", ids[s][test_rows[s]]).astype(np.float64), ref)
    FEAT_DIST[s] = nn_dist(Z_SYN[s].astype(np.float64), ref) / np.median(d_nat)
    FEAT_DIST_NAT[s] = d_nat / np.median(d_nat)
    zs, zn = Z_SYN[s], feats("dinov2", ids[s][test_rows[s]])
    feat_rows.append({"subject": s, "syn_dist_median": float(np.median(FEAT_DIST[s])), "syn_dist_p90": float(np.percentile(FEAT_DIST[s], 90)),
                      "syn_norm_over_nat": float(np.median(np.linalg.norm(zs, axis=1)) / np.median(np.linalg.norm(zn, axis=1))),
                      "syn_spread_over_nat": float(zs.std(0).sum() / zn.std(0).sum())})
    NAT_NC[s] = np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")])
    del ref
print("feature distance", feat_rows, flush=True)

rows, per_vertex = [], {}
shared_idx = np.arange(FIXED["n_shared"])
for p, s in enumerate(SUBJECTS, start=1):
    t = time.monotonic()
    others = [d for d in SUBJECTS if d != s]
    meas, snr2 = by_image(SYN_Y[s].astype(np.float64))
    half = np.random.default_rng([FIXED["half_seed"], p]).permutation(FIXED["n_images"])
    ha, hb = np.sort(half[:len(half) // 2]), np.sort(half[len(half) // 2:])
    x_syn = Z_SYN[s]
    x_nat = feats("dinov2", ids[s][test_rows[s]])
    y_nat = np.asarray(Y[s][test_rows[s]], np.float64)
    pool = pool_rows[s]
    x_pool = feats("dinov2", ids[s][pool])
    y_pool = np.asarray(Y[s][pool], np.float32)
    preds, owns = {}, []
    for kind, B_ in (("", BETA), ("_ridge", BETA_R)):  # the paper's OLS encoders, then ridge ones
        avg = {}
        for where, x in (("syn", x_syn), ("nat", x_nat), ("pool", x_pool)):
            acc = np.zeros((len(x), V), np.float32)  # running sum in float32, as 10c: the seven pool predictions at once do not fit
            for d in others:
                acc += ols_pred(B_[("dinov2", d)], x).astype(np.float32)
            avg[where] = acc / len(others)
        mean_b, basis = lm.prediction_basis(avg["pool"], FIXED["K"])
        preds["avg" + kind] = (avg["syn"], avg["nat"])
        preds["own_full" + kind] = (ols_pred(B_[("dinov2", s)], x_syn), ols_pred(B_[("dinov2", s)], x_nat))
        maps = []
        for d in FIXED["draws"]:
            idx = np.random.default_rng([d, p]).permutation(len(pool))[:FIXED["k"]]
            model = lm.fit_lowrank_map(avg["pool"][idx].astype(np.float64), y_pool[idx].astype(np.float64), mean_b, basis, seed=d)
            maps.append([lm.apply_lowrank_map(model, avg[w], mean_b, basis) for w in ("syn", "nat")])
            if not kind:
                own = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(x_pool[idx], y_pool[idx].astype(np.float64))
                owns.append([own.predict(x_syn), own.predict(x_nat)])
        preds["map" + kind] = tuple(np.mean([m[i] for m in maps], 0) for i in (0, 1))
        del acc, avg
    del x_pool
    preds["own_k"] = tuple(np.mean([o[i] for o in owns], 0) for i in (0, 1))
    # diagnostics: each synthetic image's pattern correlation (over vertices) for every method, against its distance
    # from the natural images in feature space (FEAT_DIST, computed once below)
    pat = lambda pr, m: r_vert(pr.T, m.T)
    img_rows = {name: pat(ps, meas) for name, (ps, pn) in preds.items()}
    img_rows_nat = {name: pat(pn, y_nat) for name, (ps, pn) in preds.items()}
    np.savez(C.METRICS_DIR / f"10i_images_{s}.npz", feat_dist=FEAT_DIST[s], feat_dist_nat=FEAT_DIST_NAT[s],
             **{f"syn_{k}": v for k, v in img_rows.items()}, **{f"nat_{k}": v for k, v in img_rows_nat.items()})
    out = {"ceiling": ceiling(snr2, np.arange(FIXED["n_images"])), "ceiling_a": ceiling(snr2, ha)}
    nat_ceiling = np.sqrt(np.clip(NAT_NC[s], 0, None) / 100)  # NSD's ceiling for the natural test images (percent variance, 3 trials), as r
    rec = {"subject": s, "ceiling_median": float(np.median(out["ceiling"])), "nat_ceiling_median": float(np.median(nat_ceiling)),
           **{f"{n}_pattern_vs_dist_spearman": float(stats.spearmanr(img_rows[n], FEAT_DIST[s]).statistic) for n in ("avg", "avg_ridge", "own_k")}}
    for name, (ps, pn) in preds.items():
        out[name], out[name + "_a"], out[name + "_b"] = r_vert(ps, meas), r_vert(ps[ha], meas[ha]), r_vert(ps[hb], meas[hb])
        rec[f"{name}_r_syn"] = float(out[name].mean())
        r_nat = r_vert(pn, y_nat)
        rec[f"{name}_r_nat"] = float(r_nat.mean())
        rec[f"{name}_r_syn_over_ceiling"] = float(np.mean(out[name] / np.maximum(out["ceiling"], 0.05)))
        rec[f"{name}_r_nat_over_ceiling"] = float(np.mean(r_nat / np.maximum(nat_ceiling, 0.05)))
        rec[f"{name}_top1_syn_shared"] = nc.identify_top1(ps[shared_idx], meas[shared_idx], centred=True)
        rec[f"{name}_top1_nat"] = float(np.mean([nc.identify_top1(pn[g], y_nat[g], centred=True) for g in GALS]))
    per_vertex[s] = out
    np.savez(C.METRICS_DIR / f"10i_vertices_{s}.npz", **out)
    rows.append(rec)
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "10i_people.csv", index=False)
    print(s, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rec.items()}, f"{time.monotonic() - t:.0f} s |",
          f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
def fit(gain, ceil, borrow):
    x = np.column_stack([np.ones_like(ceil), ceil, borrow])
    coef, *_ = np.linalg.lstsq(x, gain, rcond=None)
    r2 = 1 - ((gain - x @ coef) ** 2).sum() / ((gain - gain.mean()) ** 2).sum()
    return {"intercept": float(coef[0]), "ceiling": float(coef[1]), "borrowing": float(coef[2]), "r2": float(r2)}
frame = pd.DataFrame(rows).set_index("subject")
def desc(x):
    return {"mean": float(x.mean()), "sd": float(x.std(ddof=1)), "positive": int((x > 0).sum()), "n": int(len(x)),
            "per_person": {k: round(float(v), 4) for k, v in x.items()}}
res = {"post_hoc": "development on NSD-synthetic with DINOv2 features (10h's code)", "people": rows, "contrasts": {
    "map_minus_avg_r_syn": desc(frame.map_r_syn - frame.avg_r_syn),
    "map_minus_avg_r_nat": desc(frame.map_r_nat - frame.avg_r_nat),
    "own_full_minus_avg_r_syn": desc(frame.own_full_r_syn - frame.avg_r_syn),
    "own_full_minus_avg_r_nat": desc(frame.own_full_r_nat - frame.avg_r_nat),
    "map_minus_avg_top1_syn_shared": desc(frame.map_top1_syn_shared - frame.avg_top1_syn_shared),
    "map_minus_own_k_r_syn": desc(frame.map_r_syn - frame.own_k_r_syn),
    "avg_r_syn_over_nat": desc(frame.avg_r_syn / frame.avg_r_nat),
    # ridge donors
    "map_ridge_minus_avg_ridge_r_syn": desc(frame.map_ridge_r_syn - frame.avg_ridge_r_syn),
    "map_ridge_minus_avg_ridge_r_nat": desc(frame.map_ridge_r_nat - frame.avg_ridge_r_nat),
    "avg_ridge_minus_avg_r_syn": desc(frame.avg_ridge_r_syn - frame.avg_r_syn),
    "avg_ridge_minus_avg_r_nat": desc(frame.avg_ridge_r_nat - frame.avg_r_nat),
    "own_full_ridge_minus_own_full_r_syn": desc(frame.own_full_ridge_r_syn - frame.own_full_r_syn),
    "map_ridge_minus_own_k_r_syn": desc(frame.map_ridge_r_syn - frame.own_k_r_syn),
    "avg_ridge_r_syn_over_nat": desc(frame.avg_ridge_r_syn / frame.avg_ridge_r_nat),
    "avg_ceiling_ratio_syn_over_nat": desc(frame.avg_r_syn_over_ceiling / frame.avg_r_nat_over_ceiling),
    "avg_ridge_ceiling_ratio_syn_over_nat": desc(frame.avg_ridge_r_syn_over_ceiling / frame.avg_ridge_r_nat_over_ceiling),
    "avg_pattern_vs_dist_spearman": desc(frame.avg_pattern_vs_dist_spearman),
    "avg_ridge_pattern_vs_dist_spearman": desc(frame.avg_ridge_pattern_vs_dist_spearman),
    "own_k_pattern_vs_dist_spearman": desc(frame.own_k_pattern_vs_dist_spearman)}}
res["features"] = feat_rows
P = {k: np.concatenate([o[k] for o in per_vertex.values()]) for k in ("map", "avg", "ceiling", "map_b", "avg_b", "ceiling_a", "avg_a", "own_full",
                                                                      "map_ridge", "avg_ridge", "map_ridge_b", "avg_ridge_b", "avg_ridge_a")}
gain_b, head_a = P["map_b"] - P["avg_b"], P["ceiling_a"] - P["avg_a"]
res["headroom"] = {"map_fit": fit(P["map"] - P["avg"], P["ceiling"], P["avg"]), "own_fit": fit(P["own_full"] - P["avg"], P["ceiling"], P["avg"]),
                   "split_half_corr": float(np.corrcoef(gain_b, head_a)[0, 1]),
                   "split_half_per_person": {s: float(np.corrcoef(o["map_b"] - o["avg_b"], o["ceiling_a"] - o["avg_a"])[0, 1]) for s, o in per_vertex.items()}}
res["headroom"]["split_half_positive"] = int(sum(v > 0 for v in res["headroom"]["split_half_per_person"].values()))
res["headroom"]["map_ridge_fit"] = fit(P["map_ridge"] - P["avg_ridge"], P["ceiling"], P["avg_ridge"])
res["headroom"]["ridge_split_half_corr"] = float(np.corrcoef(P["map_ridge_b"] - P["avg_ridge_b"], P["ceiling_a"] - P["avg_ridge_a"])[0, 1])
(C.METRICS_DIR / "10i_summary.json").write_text(json.dumps(res, indent=2))
print("10I_SUMMARY_JSON " + json.dumps(res))